# KvForge v3: CoTo Progressive LoRA


In [ ]:
import json, math, time, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.cache_utils import DynamicCache

device = "cpu"
print("Device:", device)

class LoRAConv1D(nn.Module):
    def __init__(self, orig, r=8, alpha=16.0):
        super().__init__()
        self.orig = orig; self.scaling = alpha / r
        in_f = orig.weight.shape[0]; out_f = orig.nf
        self.lora_A = nn.Parameter(torch.randn(in_f, r) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(r, out_f))
        self.active = True
    def activate(self, a=True): self.active = a
    def forward(self, x):
        h = self.orig(x)
        if self.active: h = h + (x @ self.lora_A @ self.lora_B) * self.scaling
        return h

def inject_lora(model, r=8, alpha=16.0):
    count = 0
    for n, m in model.named_modules():
        if n.endswith(".attn.c_attn") or n.endswith(".attn.c_proj"):
            parent = model; parts = n.split("."); child = parts[-1]
            for p in parts[:-1]:
                if p: parent = getattr(parent, p)
            setattr(parent, child, LoRAConv1D(m, r=r, alpha=alpha))
            count += 1
    print("  LoRA injected:", count, "modules")
    return count

def set_lora(m, a):
    for mod in m.modules():
        if hasattr(mod, "activate"): mod.activate(a)

def progressive_activate(model, step, total_steps, schedule="linear"):
    modules = [(n,m) for n,m in model.named_modules() if hasattr(m,"activate") and hasattr(m,"lora_A")]
    n_total = len(modules)
    progress = step / max(total_steps, 1)
    if schedule == "immediate": n_active = n_total
    elif schedule == "linear": n_active = max(1, int(n_total * min(1.0, progress * 1.5)))
    elif schedule == "exponential": n_active = max(1, int(n_total * (1 - math.exp(-progress * 4))))
    elif schedule == "cosine": n_active = max(1, int(n_total * (1 - math.cos(progress * math.pi / 2))))
    else: n_active = n_total
    for i, (name, mod) in enumerate(modules):
        mod.activate(i < n_active)
    return n_active, n_total

def train_schedule(model, texts, steps, lr=3e-3, schedule="linear"):
    tok = AutoTokenizer.from_pretrained("gpt2")
    tok.pad_token = tok.eos_token
    params = [p for n,p in model.named_parameters() if "lora" in n]
    opt = torch.optim.AdamW(params, lr=lr)
    model.train()
    losses = []
    for s in range(steps):
        text = texts[s % len(texts)]
        inp = tok(text, return_tensors="pt", truncation=True, max_length=128).to(device)
        ids = inp["input_ids"]
        n_act, n_tot = progressive_activate(model, s, steps, schedule)
        out = model(ids)
        loss = F.cross_entropy(out.logits[0, :-1], ids[0, 1:])
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if s % 30 == 0:
            print("  Step %d | Active: %d/%d | Loss: %.4f" % (s, n_act, n_tot, loss.item()))
    model.eval()
    return losses

def compress_past(past_kv, bits):
    if bits >= 16: return past_kv
    dc = DynamicCache()
    for li, layer in enumerate(past_kv):
        k, v = layer[0], layer[1]
        mnk, mxk = k.min(-1, True).values, k.max(-1, True).values
        sk = (mxk - mnk).clamp(1e-8) / (2**bits - 1)
        dk = (((k - mnk) / sk).round().clamp(0, 2**bits-1).float() * sk + mnk).to(k.dtype)
        mnv, mxv = v.min(-1, True).values, v.max(-1, True).values
        sv = (mxv - mnv).clamp(1e-8) / (2**bits - 1)
        dv = (((v - mnv) / sv).round().clamp(0, 2**bits-1).float() * sv + mnv).to(v.dtype)
        dc.update(dk, dv, dk.size(2))
    return dc

def cache_mb_native(past):
    total = 0
    for layer in past:
        k, v = layer[0], layer[1]
        total += k.numel() * k.element_size() + v.numel() * v.element_size()
    return total / (1024**2)

print("=" * 60)
print("KvForge v3: CoTo Progressive LoRA")
print("=" * 60)

texts = [
    "The transformer architecture uses self-attention to process sequences in parallel.",
    "Large language models generate text by predicting the next token given the previous context.",
    "KV cache compression reduces memory usage during inference by quantizing key-value pairs.",
    "LoRA adapters fine-tune large models by learning low-rank updates to attention projections.",
]

print("\n[1/3] Training Baseline (all-LoRA)...", end=" ")
bm1 = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
inject_lora(bm1, r=8)
l1 = train_schedule(bm1, texts, steps=60, schedule="immediate")
print("Loss: %.4f -> %.4f" % (l1[0], l1[-1]))

print("[2/3] Training CoTo Linear...", end=" ")
bm2 = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
inject_lora(bm2, r=8)
l2 = train_schedule(bm2, texts, steps=60, schedule="linear")
print("Loss: %.4f -> %.4f" % (l2[0], l2[-1]))

print("[3/3] Training CoTo Cosine...", end=" ")
bm3 = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
inject_lora(bm3, r=8)
l3 = train_schedule(bm3, texts, steps=60, schedule="cosine")
print("Loss: %.4f -> %.4f" % (l3[0], l3[-1]))

# Benchmark
tok = AutoTokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token
text = "The transformer architecture revolutionized NLP by introducing self-attention."
inp = tok(text, return_tensors="pt", truncation=True, max_length=96).to(device)

def bench(name, model_obj, bits=16):
    set_lora(model_obj, False)
    t0 = time.time()
    with torch.no_grad():
        out = model_obj.generate(**inp, max_new_tokens=1, use_cache=True,
            pad_token_id=tok.eos_token_id, do_sample=False, return_dict_in_generate=True)
    past = out.past_key_values
    tp = time.time() - t0
    cm_orig = cache_mb_native(past)
    dpast = compress_past(past, bits)
    cm = cm_orig * (bits / 16.0) if bits < 16 else cm_orig
    last_tok = out.sequences[:, -1:]
    t0 = time.time()
    set_lora(model_obj, True)
    with torch.no_grad():
        for _ in range(12):
            out_d = model_obj(last_tok, past_key_values=dpast, use_cache=True)
            dpast = out_d.past_key_values
            last_tok = out_d.logits[:, -1:].argmax(dim=-1)
    td = time.time() - t0
    set_lora(model_obj, False)
    with torch.no_grad():
        out_b = model_obj(inp["input_ids"])
    ppl_off = math.exp(F.cross_entropy(out_b.logits[0, :-1], inp["input_ids"][0, 1:]).item())
    set_lora(model_obj, True)
    with torch.no_grad():
        out_b2 = model_obj(inp["input_ids"])
    ppl_on = math.exp(F.cross_entropy(out_b2.logits[0, :-1], inp["input_ids"][0, 1:]).item())
    set_lora(model_obj, False)
    return {"name": name, "prefill_ms": round(tp*1000,2), "decode_ms": round(td*1000,2),
            "cache_mb": round(cm,4), "ppl_off": round(ppl_off,4), "ppl_on": round(ppl_on,4)}

results = []
print("\n" + "=" * 60)
print("RESULTS: CoTo vs Baseline")
print("=" * 60)
print("  %-25s %6s %6s %8s %8s %8s %8s" % ("Model", "Pre", "Dec", "Cache", "PPLoff", "PPLon", "Gap"))
print("  " + "-"*79)

for name, model_obj, bits in [
    ("Baseline (all-LoRA)", bm1, 16),
    ("CoTo Linear", bm2, 16),
    ("CoTo Cosine", bm3, 16),
    ("Baseline (4-bit)", bm1, 4),
    ("CoTo Linear (4-bit)", bm2, 4),
    ("CoTo Cosine (4-bit)", bm3, 4),
]:
    r = bench(name, model_obj, bits)
    gap = r["ppl_on"] - r["ppl_off"]
    results.append({**r, "gap": round(gap,4)})
    print("  %-25s %6.1f %6.1f %8.4f %8.2f %8.2f %8.4f" %
          (name, r["prefill_ms"], r["decode_ms"], r["cache_mb"], r["ppl_off"], r["ppl_on"], gap))

print("\n" + "=" * 60)
print("GAP ANALYSIS")
print("=" * 60)
print("  Gap = PPL(on) - PPL(off): lower means smoother on/off switching")
gap_bl = results[0]["gap"]
print("  Baseline gap: %.4f" % gap_bl)
print("  CoTo Linear gap: %.4f (%.1f%% improvement)" %
      (results[1]["gap"], ((gap_bl - results[1]["gap"]) / max(gap_bl, 0.001)) * 100))
print("  CoTo Cosine gap: %.4f (%.1f%% improvement)" %
      (results[2]["gap"], ((gap_bl - results[2]["gap"]) / max(gap_bl, 0.001)) * 100))

out = {"train_loss": {
    "baseline": [round(l1[0],4), round(l1[-1],4)],
    "coto_linear": [round(l2[0],4), round(l2[-1],4)],
    "coto_cosine": [round(l3[0],4), round(l3[-1],4)],
}, "benchmarks": results}

with open("/kaggle/working/results.json", "w") as f:
    json.dump(out, f, indent=2)
print("\nresults.json saved | Done!")
